In [56]:
import pandas as pd

In [57]:
df = pd.read_csv('vehicles_raw.csv')

In [59]:
df.isnull().sum()

,0
id,0
url,0
region,1
region_url,1
price,1
year,86
manufacturer,660
model,215
condition,6204
cylinders,6215


In [60]:
drop_cols = ["id", "url", "region_url", "image_url", "description"]
df = df.drop(columns=drop_cols)

In [61]:
high_missing = ["county", "size", "VIN"]
df = df.drop(columns=high_missing)

In [62]:
cat_cols = ["condition", "cylinders", "drive", "paint_color", "type"]

In [63]:
df[cat_cols] = df[cat_cols].fillna("unknown")

In [64]:
df["manufacturer"] = df["manufacturer"].fillna(df["manufacturer"].mode()[0])
df["model"] = df["model"].fillna("unknown")
df["fuel"] = df["fuel"].fillna(df["fuel"].mode()[0])
df["title_status"] = df["title_status"].fillna(df["title_status"].mode()[0])

In [65]:
df["year"] = df["year"].fillna(df["year"].median())
df["odometer"] = df["odometer"].fillna(df["odometer"].median())

In [66]:
df = df.dropna(subset=["price"])

In [67]:
df = df.dropna(subset=["lat", "long"])

In [69]:
df["car_age"] = 2026 - df["year"]

In [70]:
df = df[df["price"] < 100000]   # remove unrealistic prices
df = df[df["price"] > 500]      # remove junk values

In [71]:
df.isnull().sum()

,0
region,0
price,0
year,0
manufacturer,0
model,0
condition,0
cylinders,0
fuel,0
odometer,0
title_status,0


In [72]:
df['transmission']

,transmission
0,automatic
1,manual
2,automatic
3,automatic
4,automatic
...,...
14995,other
14996,other
14997,automatic
14998,automatic


In [73]:
df["transmission"] = df["transmission"].fillna(df["transmission"].mode()[0])

In [74]:
df.isnull().sum()

,0
region,0
price,0
year,0
manufacturer,0
model,0
condition,0
cylinders,0
fuel,0
odometer,0
title_status,0


In [75]:
df

,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,lat,long,posting_date,car_age
0,anchorage / mat-su,23975.0,2015.0,mercedes-benz,cla250 4matic awd,unknown,unknown,gas,53348.0,clean,automatic,unknown,unknown,unknown,ak,61.573915,-149.399197,2021-04-20T12:06:12-0800,11.0
1,fayetteville,36500.0,1973.0,chevrolet,camaro z28,unknown,unknown,gas,22673.0,clean,manual,unknown,unknown,unknown,ar,36.043600,-94.253900,2021-04-12T11:07:20-0500,53.0
2,flagstaff / sedona,21495.0,2018.0,ford,mustang,good,unknown,gas,63210.0,clean,automatic,unknown,other,silver,az,35.189958,-111.666438,2021-04-17T14:00:28-0700,8.0
3,huntsville / decatur,31900.0,2016.0,gmc,sierra 3500,unknown,unknown,gas,140000.0,clean,automatic,unknown,unknown,unknown,al,35.265997,-87.322254,2021-05-01T13:41:49-0500,10.0
4,prescott,39495.0,2020.0,ram,1500,like new,8 cylinders,gas,33629.0,clean,automatic,rwd,truck,silver,az,34.586025,-112.318347,2021-04-27T20:34:00-0700,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,dothan,32990.0,2017.0,jeep,wrangler unlimited sport,good,6 cylinders,gas,30041.0,clean,other,4wd,other,silver,al,31.230000,-85.400000,2021-05-02T12:11:16-0500,9.0
14996,auburn,34590.0,2016.0,chevrolet,silverado 1500 double,good,6 cylinders,gas,29499.0,clean,other,4wd,pickup,silver,al,32.590000,-85.480000,2021-05-03T12:41:33-0500,10.0
14997,fresno / madera,23999.0,2007.0,toyota,fj cruiser,unknown,6 cylinders,gas,113000.0,clean,automatic,4wd,SUV,grey,ca,38.614287,-121.270495,2021-04-22T12:03:39-0700,19.0
14998,fresno / madera,10995.0,2013.0,ford,fusion,excellent,4 cylinders,hybrid,110155.0,clean,automatic,fwd,sedan,red,ca,38.611926,-121.423565,2021-04-22T15:37:20-0700,13.0


In [76]:
df.shape

(13460, 19)

In [77]:
# Convert everything to string first
df["cylinders"] = df["cylinders"].astype(str)

# Extract only numbers from entries like "8 cylinders"
df["cylinders"] = df["cylinders"].str.extract(r'(\d+)')

# Replace NaN (which came from 'unknown') with 'UNKNOWN'
df["cylinders"] = df["cylinders"].fillna("UNKNOWN")

In [78]:
df = df.replace("unknown", "UNKNOWN")

In [79]:
df["year"] = df["year"].astype(int)
df["car_age"] = df["car_age"].astype(int)

In [80]:
df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce", utc=True)

In [81]:
df["post_year"] = df["posting_date"].dt.year
df["post_month"] = df["posting_date"].dt.month

In [82]:
df["price"].describe()

,price
count,13460.000000
mean,21774.033581
std,14774.347101
min,507.000000
25%,9995.000000
50%,18995.000000
75%,29997.000000
max,99950.000000


In [83]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df = df[(df["price"] >= lower_bound) & (df["price"] <= upper_bound)]

In [84]:
df.to_csv('vehicles_cleaned.csv',index=False)

In [85]:
df.isnull().sum()

,0
region,0
price,0
year,0
manufacturer,0
model,0
condition,0
cylinders,0
fuel,0
odometer,0
title_status,0


In [86]:
df.dtypes

,0
region,object
price,float64
year,int64
manufacturer,object
model,object
condition,object
cylinders,object
fuel,object
odometer,float64
title_status,object


In [87]:
df["price"].describe()

,price
count,13190.000000
mean,20773.972934
std,13093.663075
min,507.000000
25%,9950.000000
50%,18753.000000
75%,29990.000000
max,60000.000000


In [88]:
df["cylinders"].unique()

array(['UNKNOWN', '8', '6', '4', '3', '10', '5', '12'], dtype=object)

In [89]:
for col in df.select_dtypes(include="object").columns:
    print(col, df[col].nunique())

region 30
manufacturer 38
model 3714
condition 7
cylinders 8
fuel 5
title_status 6
transmission 3
drive 4
type 14
paint_color 13
state 5


In [90]:
df.shape

(13190, 21)